## 1. Importing + Constants

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
ORIGINAL_DATA_PATH = "../original-data"
PROCESSED_DATA_PATH = "../processed-data"

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)


## 2. Preprocessing Data

In [3]:
# 1. Tải dữ liệu gốc
sales = pd.read_csv(f'{ORIGINAL_DATA_PATH}/sales.csv', parse_dates=['Date'])
submission = pd.read_csv(f'{ORIGINAL_DATA_PATH}/sample_submission.csv', parse_dates=['Date'])
promotions = pd.read_csv(f'{ORIGINAL_DATA_PATH}/promotions.csv', parse_dates=['start_date', 'end_date'])

# 2. Tạo trục thời gian toàn vẹn (từ Train đến hết Test)
all_dates = pd.date_range(start=sales['Date'].min(), end=submission['Date'].max(), freq='D')
df = pd.DataFrame({'Date': all_dates})

# Đánh dấu tập Train/Test
df = df.merge(sales, on='Date', how='left')
df['Split'] = np.where(df['Date'] <= sales['Date'].max(), 'Train', 'Test')

# 3. Kỹ thuật đặc trưng Thời gian (Calendar Features)
def create_calendar_features(df):
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    df['quarter'] = df['Date'].dt.quarter
    df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['Date'].dt.is_month_end.astype(int)
    
    # Mã hóa vòng (Cyclical encoding) giúp mô hình hiểu tính chu kỳ
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)
    
    return df

df = create_calendar_features(df)

# 4. Thêm các ngày lễ Việt Nam
def add_vietnam_holidays(df):
    # Các ngày lễ cố định
    df['is_holiday'] = 0
    # Tết Dương Lịch
    df.loc[(df['month'] == 1) & (df['day'] == 1), 'is_holiday'] = 1
    # Giải phóng miền Nam (30/04) & Quốc tế Lao động (01/05)
    df.loc[(df['month'] == 4) & (df['day'] == 30), 'is_holiday'] = 1
    df.loc[(df['month'] == 5) & (df['day'] == 1), 'is_holiday'] = 1
    # Quốc khánh (02/09)
    df.loc[(df['month'] == 9) & (df['day'] == 2), 'is_holiday'] = 1
    
    # Lưu ý: Với Tết Nguyên Đán (Âm lịch), bạn nên tạo một danh sách các ngày cụ thể 
    # từ 2012-2024 vì nó thay đổi hàng năm.
    tet_dates = [
        '2022-01-31', '2022-02-01', '2022-02-02', # 2022
        '2023-01-21', '2023-01-22', '2023-01-23', # 2023
        '2024-02-09', '2024-02-10', '2024-02-11'  # 2024
    ]
    df['is_tet'] = df['Date'].dt.strftime('%Y-%m-%d').isin(tet_dates).astype(int)
    return df

df = add_vietnam_holidays(df)

# 5. Xử lý Khuyến mãi (Chỉ tính những biến biết trước trong tương lai)
def count_active_promos(current_date):
    active = promotions[(promotions['start_date'] <= current_date) & (promotions['end_date'] >= current_date)]
    return len(active)

df['active_promos'] = df['Date'].apply(count_active_promos)

# 6. Tạo Target Lags (Cho tập Train)
# Quan trọng: Trong tập Test, các biến này sẽ được điền bằng phương pháp dự báo đệ quy (Recursive)
# Ở bước preprocessing này, chúng ta chỉ tạo cột để mô hình học.
lags = [1, 7, 14, 30, 365]
for lag in lags:
    df[f'rev_lag_{lag}'] = df['Revenue'].shift(lag)
    df[f'cogs_lag_{lag}'] = df['COGS'].shift(lag)

# 7. Lưu dữ liệu
# Loại bỏ các dòng đầu tiên bị NaN do cơ chế Lag
df = df[df['Date'] >= (sales['Date'].min() + pd.Timedelta(days=365))]

df.to_csv(f'{PROCESSED_DATA_PATH}/final_training_data.csv', index=False)
print("Đã cập nhật xong dữ liệu Preprocessing.")

Đã cập nhật xong dữ liệu Preprocessing.
